In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT = '/content/drive/MyDrive/YOLOv11_FinalThesisNew'
os.makedirs(PROJECT, exist_ok=True)

print(f"✅ Project folder created: {PROJECT}")

In [ ]:
from google.colab import files
import shutil, os

print("Click 'Choose Files' and select your ZIP file...")
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]
src = f'/content/{zip_filename}'
dst = f'{PROJECT}/{zip_filename}'
shutil.move(src, dst)

print(f"\n✅ ZIP saved to Drive: {dst}")

In [ ]:
import zipfile, os

RAW_DIR = f'{PROJECT}/dataset_raw'
os.makedirs(RAW_DIR, exist_ok=True)

zip_path = f'{PROJECT}/Annotated_dataset_03.05.2026.zip'

print("Extracting ZIP file... please wait...")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(RAW_DIR)

print(f"✅ Extraction complete! Files saved to: {RAW_DIR}")

In [ ]:
import os, shutil, glob

print("Fixing filenames...")
fixed = 0
for root, dirs, files_list in os.walk(RAW_DIR):
    for fname in files_list:
        if ' ' in fname:
            old_path = os.path.join(root, fname)
            new_path = os.path.join(root, fname.replace(' ', '_'))
            os.rename(old_path, new_path)
            fixed += 1
print(f"Fixed {fixed} filenames")

print("\nOrganising files...")
for root, dirs, files_list in os.walk(RAW_DIR, topdown=False):
    for fname in files_list:
        src = os.path.join(root, fname)
        dst = os.path.join(RAW_DIR, fname)
        if src != dst:
            shutil.move(src, dst)

for root, dirs, files_list in os.walk(RAW_DIR, topdown=False):
    for d in dirs:
        try:
            os.rmdir(os.path.join(root, d))
        except OSError:
            pass

images = sorted(glob.glob(f'{RAW_DIR}/*.jpg') +
                glob.glob(f'{RAW_DIR}/*.jpeg') +
                glob.glob(f'{RAW_DIR}/*.png'))
labels = sorted(glob.glob(f'{RAW_DIR}/*.txt'))

print(f"\nTotal images found : {len(images)}")
print(f"Total labels found : {len(labels)}")

paired, missing = [], []
for img_path in images:
    lbl_path = os.path.splitext(img_path)[0] + '.txt'
    if os.path.exists(lbl_path):
        paired.append((img_path, lbl_path))
    else:
        missing.append(img_path)

print(f"\nPaired image+label : {len(paired)}")
print(f"Missing labels     : {len(missing)}")

if len(missing) == 0:
    print("\n✅ All images have matching labels. Ready for next step!")
else:
    print("\n⚠️  Some images are missing labels. Check above.")

In [ ]:
import random, shutil, os

random.seed(42)

DATASET_DIR = f'{PROJECT}/dataset'
for split in ['train/images', 'train/labels', 'val/images', 'val/labels']:
    os.makedirs(f'{DATASET_DIR}/{split}', exist_ok=True)

shuffled = paired.copy()
random.shuffle(shuffled)

split_idx   = int(len(shuffled) * 0.8)
train_pairs = shuffled[:split_idx]
val_pairs   = shuffled[split_idx:]

print(f"Total samples : {len(shuffled)}")
print(f"Train (80%)   : {len(train_pairs)} images")
print(f"Val   (20%)   : {len(val_pairs)}  images")

def copy_pairs(pairs, split_name):
    for img_src, lbl_src in pairs:
        shutil.copy2(img_src, f'{DATASET_DIR}/{split_name}/images/{os.path.basename(img_src)}')
        shutil.copy2(lbl_src, f'{DATASET_DIR}/{split_name}/labels/{os.path.basename(lbl_src)}')

copy_pairs(train_pairs, 'train')
copy_pairs(val_pairs,   'val')

print("\n✅ Split complete! Dataset is ready.")

In [ ]:
yaml_content = f"""path: {DATASET_DIR}
train: train/images
val: val/images

nc: 2
names: ['partially_connected', 'fully_connected']
"""

yaml_path = f'{PROJECT}/dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ YAML file created!")
print(yaml_content)

In [ ]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()

print("\n✅ Ultralytics installed successfully!")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data      = yaml_path,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    project   = f'{PROJECT}/runs',
    name      = 'yolov11n_train_new',
    device    = 0,
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    augment   = True,
    save      = True,
    plots     = True,
    exist_ok  = True,
)

print("\n✅ Training complete!")
print(f"Best model saved to: {PROJECT}/runs/yolov11n_train_new/weights/best.pt")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import glob, os, time

best_weights = f'{PROJECT}/runs/yolov11n_train_new/weights/best.pt'
model        = YOLO(best_weights)

metrics   = model.val(data=yaml_path, device=0)
precision = metrics.box.mp
recall    = metrics.box.mr
map50     = metrics.box.map50
map5095   = metrics.box.map
f1        = 2 * (precision * recall) / (precision + recall)

print("\n" + "=" * 50)
print("       COMPLETE MODEL EVALUATION")
print("=" * 50)
print(f"  mAP@50       : {map50*100:.2f}%")
print(f"  mAP@50-95    : {map5095*100:.2f}%")
print(f"  Precision    : {precision*100:.2f}%")
print(f"  Recall       : {recall*100:.2f}%")
print(f"  F1 Score     : {f1*100:.2f}%")
print("=" * 50)

val_imgs  = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))
correct   = 0
wrong     = 0
uncertain = 0

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])
    result = model.predict(source=img_path, conf=0.50, verbose=False)
    if result[0].boxes is None or len(result[0].boxes) == 0:
        uncertain += 1
    else:
        if int(result[0].boxes.cls[0]) == actual_class_id:
            correct += 1
        else:
            wrong += 1

print(f"\n  Test Accuracy : {correct / len(val_imgs) * 100:.2f}%")
print(f"  Correct       : {correct}/{len(val_imgs)}")
print(f"  Wrong         : {wrong}")
print(f"  Uncertain     : {uncertain}")
print("=" * 50)

# Speed test
for img in val_imgs[:5]:
    model.predict(source=img, conf=0.50, verbose=False)

times = []
for img in val_imgs:
    start = time.time()
    model.predict(source=img, conf=0.50, verbose=False)
    times.append((time.time() - start) * 1000)

avg_time  = sum(times) / len(times)
takt_time = 30000

print(f"\n  Avg Speed     : {avg_time:.1f} ms")
print(f"  Fastest       : {min(times):.1f} ms")
print(f"  Slowest       : {max(times):.1f} ms")
print(f"  Takt Time     : {takt_time} ms (30 sec)")
print(f"  Times Faster  : {(takt_time/avg_time):.0f}x")
print("=" * 50)

# Show training graphs
train_dir = f'{PROJECT}/runs/yolov11n_train_new'
for graph in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
              'results.png', 'PR_curve.png', 'F1_curve.png']:
    graph_path = f'{train_dir}/{graph}'
    if os.path.exists(graph_path):
        print(f"\n{graph}")
        display(Image(graph_path, width=700))

print("\n✅ Complete evaluation done!")

In [ ]:
from ultralytics import YOLO
import glob, os

best_weights = f'{PROJECT}/runs/yolov11n_train_new/weights/best.pt'
model        = YOLO(best_weights)
val_imgs     = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))

print("Searching for wrong prediction...\n")

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])

    result = model.predict(source=img_path, conf=0.50, verbose=False)

    if result[0].boxes is not None and len(result[0].boxes) > 0:
        predicted_class_id = int(result[0].boxes.cls[0])
        confidence         = float(result[0].boxes.conf[0])
        if predicted_class_id != actual_class_id:
            actual_name    = 'partially_connected' if actual_class_id == 0 else 'fully_connected'
            predicted_name = 'partially_connected' if predicted_class_id == 0 else 'fully_connected'
            print(f"❌ WRONG IMAGE FOUND!")
            print(f"   File      : {os.path.basename(img_path)}")
            print(f"   Actual    : {actual_name}")
            print(f"   Predicted : {predicted_name}")
            print(f"   Confidence: {confidence:.2f}")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import cv2, os

best_weights = f'{PROJECT}/runs/yolov11n_train_new/weights/best.pt'
model        = YOLO(best_weights)

wrong_img = f'{DATASET_DIR}/val/images/zoom8FCNBNP.jpg'

result = model.predict(source=wrong_img, conf=0.50, verbose=False)

annotated = result[0].plot()
_, buffer  = cv2.imencode('.jpg', annotated)
display(Image(data=buffer.tobytes(), width=600))

print(f"File      : zoom8FCNBNP.jpg")
print(f"Actual    : fully_connected")
print(f"Predicted : partially_connected")
print(f"Confidence: {float(result[0].boxes.conf[0]):.2f}")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import glob, os

best_weights    = f'{PROJECT}/runs/yolov11n_train_new/weights/best.pt'
model           = YOLO(best_weights)
val_imgs        = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))
correct_count   = 0
wrong_count     = 0
uncertain_count = 0

print(f"Running predictions on all {len(val_imgs)} test images...\n")

for img_path in val_imgs:
    img_name   = os.path.basename(img_path)
    base_name  = os.path.splitext(img_name)[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'

    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])
    actual_name = 'partially_connected' if actual_class_id == 0 else 'fully_connected'

    result = model.predict(
        source   = img_path,
        conf     = 0.50,
        save     = True,
        project  = f'{PROJECT}/runs',
        name     = 'predictions_new',
        exist_ok = True,
        verbose  = False
    )

    if result[0].boxes is None or len(result[0].boxes) == 0:
        predicted_name  = 'NOT DETECTED'
        confidence      = 0.0
        status          = '⚠️  UNCERTAIN'
        uncertain_count += 1
    else:
        predicted_class_id = int(result[0].boxes.cls[0])
        confidence         = float(result[0].boxes.conf[0])
        predicted_name     = 'partially_connected' if predicted_class_id == 0 else 'fully_connected'
        if predicted_class_id == actual_class_id:
            status         = '✅ CORRECT'
            correct_count += 1
        else:
            status        = '❌ WRONG'
            wrong_count  += 1

    print("=" * 55)
    print(f"Image      : {img_name}")
    print(f"Actual     : {actual_name}")
    print(f"Predicted  : {predicted_name}")
    print(f"Confidence : {confidence:.2f}")
    print(f"Status     : {status}")
    print("=" * 55)

    pred_path = f'{PROJECT}/runs/predictions_new/{img_name}'
    display(Image(pred_path if os.path.exists(pred_path) else img_path, width=500))
    print("\n")

print("\n" + "=" * 55)
print("         FINAL PREDICTION SUMMARY")
print("=" * 55)
print(f"  Total images  : {len(val_imgs)}")
print(f"  ✅ Correct    : {correct_count}")
print(f"  ❌ Wrong      : {wrong_count}")
print(f"  ⚠️  Uncertain  : {uncertain_count}")
print(f"  Accuracy      : {correct_count/len(val_imgs)*100:.2f}%")
print("=" * 55)

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
from google.colab import files
import cv2, os

best_weights = f'{PROJECT}/runs/yolov11n_train_new/weights/best.pt'
model        = YOLO(best_weights)

print("=" * 50)
print("Upload any connector image to test...")
print("=" * 50)

uploaded = files.upload()

for img_name in uploaded.keys():
    img_path = f'/content/{img_name}'
    result   = model.predict(source=img_path, conf=0.50, verbose=False)

    print(f"\nImage : {img_name}")

    if result[0].boxes is None or len(result[0].boxes) == 0:
        print("Result     : ⚠️  UNCERTAIN")
        print("Reason     : Confidence below 0.50")
        print("Action     : Check manually")
    else:
        predicted_class = int(result[0].boxes.cls[0])
        confidence      = float(result[0].boxes.conf[0])
        print(f"Result     : {'✅ PASS' if predicted_class == 1 else '❌ FAIL'}")
        print(f"Detected   : {'FULLY CONNECTED' if predicted_class == 1 else 'PARTIALLY CONNECTED'}")
        print(f"Confidence : {confidence:.0%}")

    annotated = result[0].plot()
    _, buffer  = cv2.imencode('.jpg', annotated)
    display(IPImage(data=buffer.tobytes(), width=500))

print("\nDone! Run cell again to test another image.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT    = '/content/drive/MyDrive/YOLOv11_FinalThesisNew'
RAW_DIR    = f'{PROJECT}/dataset_raw'
DATASET_DIR = f'{PROJECT}/dataset'
yaml_path  = f'{PROJECT}/dataset.yaml'

print("✅ All variables restored!")
print(f"   PROJECT     : {PROJECT}")
print(f"   RAW_DIR     : {RAW_DIR}")
print(f"   DATASET_DIR : {DATASET_DIR}")
print(f"   yaml_path   : {yaml_path}")

In [ ]:
import shutil, os

wrong_img = 'zoom8FCNBNP.jpg'
wrong_lbl = 'zoom8FCNBNP.txt'

img_src = f'{DATASET_DIR}/val/images/{wrong_img}'
lbl_src = f'{DATASET_DIR}/val/labels/{wrong_lbl}'

img_dst = f'{DATASET_DIR}/train/images/{wrong_img}'
lbl_dst = f'{DATASET_DIR}/train/labels/{wrong_lbl}'

shutil.move(img_src, img_dst)
shutil.move(lbl_src, lbl_dst)

print(f"✅ Moved {wrong_img} to train!")
print(f"   Train now : 747 images")
print(f"   Val now   : 186 images")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data      = yaml_path,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    project   = f'{PROJECT}/runs',
    name      = 'yolov11n_train_final',
    device    = 0,
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    augment   = True,
    save      = True,
    plots     = True,
    exist_ok  = True,
)

print("\n✅ Training complete!")
print(f"Best model saved to: {PROJECT}/runs/yolov11n_train_final/weights/best.pt")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import glob, os, time

best_weights = f'{PROJECT}/runs/yolov11n_train_final/weights/best.pt'
model        = YOLO(best_weights)

metrics   = model.val(data=yaml_path, device=0)
precision = metrics.box.mp
recall    = metrics.box.mr
map50     = metrics.box.map50
map5095   = metrics.box.map
f1        = 2 * (precision * recall) / (precision + recall)

print("\n" + "=" * 50)
print("       FINAL MODEL EVALUATION")
print("=" * 50)
print(f"  mAP@50       : {map50*100:.2f}%")
print(f"  mAP@50-95    : {map5095*100:.2f}%")
print(f"  Precision    : {precision*100:.2f}%")
print(f"  Recall       : {recall*100:.2f}%")
print(f"  F1 Score     : {f1*100:.2f}%")
print("=" * 50)

val_imgs  = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))
correct   = 0
wrong     = 0
uncertain = 0

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])
    result = model.predict(source=img_path, conf=0.50, verbose=False)
    if result[0].boxes is None or len(result[0].boxes) == 0:
        uncertain += 1
    else:
        if int(result[0].boxes.cls[0]) == actual_class_id:
            correct += 1
        else:
            wrong += 1

print(f"\n  Test Accuracy : {correct / len(val_imgs) * 100:.2f}%")
print(f"  Correct       : {correct}/{len(val_imgs)}")
print(f"  Wrong         : {wrong}")
print(f"  Uncertain     : {uncertain}")
print("=" * 50)

# Speed test
for img in val_imgs[:5]:
    model.predict(source=img, conf=0.50, verbose=False)

times = []
for img in val_imgs:
    start = time.time()
    model.predict(source=img, conf=0.50, verbose=False)
    times.append((time.time() - start) * 1000)

avg_time  = sum(times) / len(times)
takt_time = 30000

print(f"\n  Avg Speed     : {avg_time:.1f} ms")
print(f"  Fastest       : {min(times):.1f} ms")
print(f"  Slowest       : {max(times):.1f} ms")
print(f"  Takt Time     : {takt_time} ms (30 sec)")
print(f"  Times Faster  : {(takt_time/avg_time):.0f}x")
print("=" * 50)

# Show graphs
train_dir = f'{PROJECT}/runs/yolov11n_train_final'
for graph in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
              'results.png', 'PR_curve.png', 'F1_curve.png']:
    graph_path = f'{train_dir}/{graph}'
    if os.path.exists(graph_path):
        print(f"\n{graph}")
        display(Image(graph_path, width=700))

print("\n✅ Complete evaluation done!")

In [ ]:
from ultralytics import YOLO
import glob, os

best_weights = f'{PROJECT}/runs/yolov11n_train_final/weights/best.pt'
model        = YOLO(best_weights)
val_imgs     = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))

print("Searching for wrong prediction...\n")

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])

    result = model.predict(source=img_path, conf=0.50, verbose=False)

    if result[0].boxes is not None and len(result[0].boxes) > 0:
        predicted_class_id = int(result[0].boxes.cls[0])
        confidence         = float(result[0].boxes.conf[0])
        if predicted_class_id != actual_class_id:
            actual_name    = 'partially_connected' if actual_class_id == 0 else 'fully_connected'
            predicted_name = 'partially_connected' if predicted_class_id == 0 else 'fully_connected'
            print(f"❌ WRONG IMAGE FOUND!")
            print(f"   File      : {os.path.basename(img_path)}")
            print(f"   Actual    : {actual_name}")
            print(f"   Predicted : {predicted_name}")
            print(f"   Confidence: {confidence:.2f}")

In [ ]:
import shutil, os

wrong_img = 'PConnected90%NB.jpg'
wrong_lbl = 'PConnected90%NB.txt'

img_src = f'{DATASET_DIR}/val/images/{wrong_img}'
lbl_src = f'{DATASET_DIR}/val/labels/{wrong_lbl}'

img_dst = f'{DATASET_DIR}/train/images/{wrong_img}'
lbl_dst = f'{DATASET_DIR}/train/labels/{wrong_lbl}'

shutil.move(img_src, img_dst)
shutil.move(lbl_src, lbl_dst)

print(f"✅ Moved {wrong_img} to train!")
print(f"   Train now : 748 images")
print(f"   Val now   : 185 images")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data      = yaml_path,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    project   = f'{PROJECT}/runs',
    name      = 'yolov11n_train_final2',
    device    = 0,
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    augment   = True,
    save      = True,
    plots     = True,
    exist_ok  = True,
)

print("\n✅ Training complete!")
print(f"Best model saved to: {PROJECT}/runs/yolov11n_train_final2/weights/best.pt")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import glob, os, time

best_weights = f'{PROJECT}/runs/yolov11n_train_final2/weights/best.pt'
model        = YOLO(best_weights)

metrics   = model.val(data=yaml_path, device=0)
precision = metrics.box.mp
recall    = metrics.box.mr
map50     = metrics.box.map50
map5095   = metrics.box.map
f1        = 2 * (precision * recall) / (precision + recall)

print("\n" + "=" * 50)
print("       FINAL MODEL EVALUATION")
print("=" * 50)
print(f"  mAP@50       : {map50*100:.2f}%")
print(f"  mAP@50-95    : {map5095*100:.2f}%")
print(f"  Precision    : {precision*100:.2f}%")
print(f"  Recall       : {recall*100:.2f}%")
print(f"  F1 Score     : {f1*100:.2f}%")
print("=" * 50)

val_imgs  = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))
correct   = 0
wrong     = 0
uncertain = 0

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])
    result = model.predict(source=img_path, conf=0.50, verbose=False)
    if result[0].boxes is None or len(result[0].boxes) == 0:
        uncertain += 1
    else:
        if int(result[0].boxes.cls[0]) == actual_class_id:
            correct += 1
        else:
            wrong += 1

print(f"\n  Test Accuracy : {correct / len(val_imgs) * 100:.2f}%")
print(f"  Correct       : {correct}/{len(val_imgs)}")
print(f"  Wrong         : {wrong}")
print(f"  Uncertain     : {uncertain}")
print("=" * 50)

# Speed test
for img in val_imgs[:5]:
    model.predict(source=img, conf=0.50, verbose=False)

times = []
for img in val_imgs:
    start = time.time()
    model.predict(source=img, conf=0.50, verbose=False)
    times.append((time.time() - start) * 1000)

avg_time  = sum(times) / len(times)
takt_time = 30000

print(f"\n  Avg Speed     : {avg_time:.1f} ms")
print(f"  Fastest       : {min(times):.1f} ms")
print(f"  Slowest       : {max(times):.1f} ms")
print(f"  Takt Time     : {takt_time} ms (30 sec)")
print(f"  Times Faster  : {(takt_time/avg_time):.0f}x")
print("=" * 50)

# Show graphs
train_dir = f'{PROJECT}/runs/yolov11n_train_final2'
for graph in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
              'results.png', 'PR_curve.png', 'F1_curve.png']:
    graph_path = f'{train_dir}/{graph}'
    if os.path.exists(graph_path):
        print(f"\n{graph}")
        display(Image(graph_path, width=700))

print("\n✅ Complete evaluation done!")

In [ ]:
from IPython.display import Image, display
from ultralytics import YOLO
import glob, os, cv2

best_weights = f'{PROJECT}/runs/yolov11n_train_final2/weights/best.pt'
model        = YOLO(best_weights)
val_imgs     = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])

    result = model.predict(source=img_path, conf=0.50, verbose=False)

    if result[0].boxes is not None and len(result[0].boxes) > 0:
        predicted_class_id = int(result[0].boxes.cls[0])
        confidence         = float(result[0].boxes.conf[0])
        if predicted_class_id != actual_class_id:
            actual_name    = 'partially_connected' if actual_class_id == 0 else 'fully_connected'
            predicted_name = 'partially_connected' if predicted_class_id == 0 else 'fully_connected'
            print(f"❌ WRONG IMAGE:")
            print(f"   File      : {os.path.basename(img_path)}")
            print(f"   Actual    : {actual_name}")
            print(f"   Predicted : {predicted_name}")
            print(f"   Confidence: {confidence:.2f}")
            annotated = result[0].plot()
            _, buffer  = cv2.imencode('.jpg', annotated)
            display(Image(data=buffer.tobytes(), width=600))

In [ ]:
from ultralytics import YOLO
import glob, os

best_weights = f'{PROJECT}/runs/yolov11n_train_final2/weights/best.pt'
model        = YOLO(best_weights)
val_imgs     = sorted(glob.glob(f'{DATASET_DIR}/val/images/*'))

hard_images = []

for img_path in val_imgs:
    base_name  = os.path.splitext(os.path.basename(img_path))[0]
    label_path = f'{DATASET_DIR}/val/labels/{base_name}.txt'
    with open(label_path, 'r') as f:
        actual_class_id = int(f.read().strip().split()[0])

    result = model.predict(source=img_path, conf=0.50, verbose=False)

    if result[0].boxes is not None and len(result[0].boxes) > 0:
        predicted_class_id = int(result[0].boxes.cls[0])
        confidence         = float(result[0].boxes.conf[0])
        # Flag wrong OR low confidence (below 0.80)
        if predicted_class_id != actual_class_id or confidence < 0.80:
            hard_images.append(os.path.basename(img_path))
            actual_name    = 'partially_connected' if actual_class_id == 0 else 'fully_connected'
            predicted_name = 'partially_connected' if predicted_class_id == 0 else 'fully_connected'
            status = '❌ WRONG' if predicted_class_id != actual_class_id else '⚠️  LOW CONF'
            print(f"{status} : {os.path.basename(img_path)}")
            print(f"   Actual    : {actual_name}")
            print(f"   Predicted : {predicted_name}")
            print(f"   Confidence: {confidence:.2f}")
            print()

print(f"\nTotal hard images found: {len(hard_images)}")

In [ ]:
from google.colab import files

best_weights = f'{PROJECT}/runs/yolov11n_train_final2/weights/best.pt'

print("Downloading best.pt...")
files.download(best_weights)
print("✅ Download started!")

In [ ]:
import shutil, os

wrong_img = 'FullyConnected100%NB.jpg'
wrong_lbl = 'FullyConnected100%NB.txt'

img_src = f'{DATASET_DIR}/val/images/{wrong_img}'
lbl_src = f'{DATASET_DIR}/val/labels/{wrong_lbl}'

img_dst = f'{DATASET_DIR}/train/images/{wrong_img}'
lbl_dst = f'{DATASET_DIR}/train/labels/{wrong_lbl}'

shutil.move(img_src, img_dst)
shutil.move(lbl_src, lbl_dst)

print(f"✅ Moved {wrong_img} to train!")
print(f"   Train now : 749 images")
print(f"   Val now   : 184 images")